# Reading from CSV FILE

In [0]:
from Config import data,Data

# Write it to Delta Table

In [0]:
import requests
import pandas as pd
import argparse

def extract(path,delta_path):
    
    url = f"https://data.cityofchicago.org/resource/{path}"
    last_crash_date = "1900-10-13T00:00:00.000"
    chunk_size =150000
    offset =0
    first_chunk = True  
            
    while True:
        params = {
        "$limit": chunk_size,
        "$offset": offset,
        "$$app_token": "YwIAZkmjAhaYhdDM2hIYhHOuu",
        "$where": f"CRASH_DATE > '{last_crash_date}'",
            "$order": "CRASH_DATE ASC"
    }
        try:
            response=requests.get(url,params=params, timeout=500)
            response.raise_for_status()
            json_data = response.json()
            if not json_data:
                print("No more data to extract.")
                break
            df = pd.DataFrame(json_data)
            df_chunk = df
            spark_df_chunk = spark.createDataFrame(df_chunk)

            spark_df_chunk.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(f"abfss://source@datalakeimewore.dfs.core.windows.net/ChicagoCrashes/bronze/{delta_path}")
            # first_chunk = False

            print(f"{offset + len(df_chunk)} records written to Delta")
            offset += chunk_size

        except requests.exceptions.RequestException as err:
            print(f"An error occurred: {err}")
            break
        except Exception as e:
            print(f'An unexpected error occurred: {e}')
            break


for items in Data:
  
  url_path = items['url_path']
  delta_path = items['delta_path']

  if delta_path == 'vehicles':
    extract(url_path,delta_path)





    

## Create Table for medallion architecture

In [0]:
%sql
CREATE DATABASE bronze_layer;|

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bronze_layer.crashes_table
USING DELTA
LOCATION 'abfss://source@datalakeimewore.dfs.core.windows.net/ChicagoCrashes/bronze/crashes'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bronze_layer.persons_table
USING DELTA
LOCATION 'abfss://source@datalakeimewore.dfs.core.windows.net/ChicagoCrashes/bronze/persons'

In [0]:
%sql
DROP TABLE bronze_layer.vehicles_table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bronze_layer.vehicles_table
USING DELTA
LOCATION 'abfss://source@datalakeimewore.dfs.core.windows.net/ChicagoCrashes/bronze/vehicles'

In [0]:
for item in data:
  base_path = item['path']
  table_name = item['table']

  df = spark.read.option("header", True) \
    .option('inferSchema', True) \
    .csv(f"{base_path}")

  # Rename "BAC_RESULT VALUE" to "BAC_RESULT_VALUE" if present
  if 'BAC_RESULT VALUE' in df.columns:
    df = df.withColumnRenamed('BAC_RESULT VALUE', 'BAC_RESULT_VALUE')

  df.write.mode("overwrite").saveAsTable(f"workspace.raw_crashes.{table_name}")

  print(f"Table {table_name} created")